## Data Cleaning

In [26]:
import pandas as pd

data_path= "../data/raw/csv_files"

# reading all the csv files
customers= pd.read_csv(data_path + "/olist_customers_dataset.csv")
geolocation= pd.read_csv(data_path + "/olist_geolocation_dataset.csv")
order_items= pd.read_csv(data_path + "/olist_order_items_dataset.csv")
order_payments= pd.read_csv(data_path + "/olist_order_payments_dataset.csv")
order_reviews= pd.read_csv(data_path + "/olist_order_reviews_dataset.csv")
orders= pd.read_csv(data_path + "/olist_orders_dataset.csv")
products= pd.read_csv(data_path + "/olist_products_dataset.csv")
sellers= pd.read_csv(data_path + "/olist_sellers_dataset.csv")
category_translation= pd.read_csv(data_path + "/product_category_name_translation.csv")

In [27]:
# making a dictionary of all the files to do certain operations combinely
dfs={"customers":customers,
     "geolocation":geolocation,
     "order_items": order_items,
     "order_payments": order_payments,
     "order_reviews": order_reviews,
     "orders": orders,
     "products": products,
     "sellers": sellers,
     "category_translation": category_translation}

#### Duplicates handling

In [30]:
# removing duplicates from geolocation table
geolocation= geolocation.drop_duplicates()

# to check whether the duplicates are dropped or not, use this code
geolocation.duplicated().sum()

np.int64(0)

In [ ]:
#duplicates for this zip_code_prefix column
geolocation['geolocation_zip_code_prefix'].duplicated().sum()

np.int64(719317)

In [32]:
# Making geolocation_zip_code_prefix as unique by aggregating the lat and long columns to avoid duplicate rows while joining
geolocation =geolocation.groupby('geolocation_zip_code_prefix')[['geolocation_lat','geolocation_lng']].mean().reset_index()

# now duplicates for this column is removed using the groupby. Now this column can be used a primary key for geolocation
geolocation['geolocation_zip_code_prefix'].duplicated().sum()

np.int64(0)

#### Data type conversion

In [44]:
# data type conversion for date columns
# creating date columns for different tables
orders_date_cols=['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
                   'order_delivered_customer_date','order_estimated_delivery_date']
order_reviews_date_cols=['review_creation_date','review_answer_timestamp']

def convert_to_datetime(df, cols):
    nulls_before = df[cols].isnull().sum()
    df[cols] = df[cols].apply(pd.to_datetime)
    nulls_after = df[cols].isnull().sum()
    return pd.DataFrame({'nulls_before': nulls_before, 'nulls_after': nulls_after, 'dtype':df[cols].dtypes})

date_check = pd.concat({
    'orders': convert_to_datetime(orders, orders_date_cols),
    'order_items': convert_to_datetime(order_items, ['shipping_limit_date']),
    'order_reviews': convert_to_datetime(order_reviews, order_reviews_date_cols),
})
date_check

nulls_before  nulls_after  \
orders        order_purchase_timestamp                  0            0   
              order_approved_at                       160          160   
              order_delivered_carrier_date           1783         1783   
              order_delivered_customer_date          2965         2965   
              order_estimated_delivery_date             0            0   
order_items   shipping_limit_date                       0            0   
order_reviews review_creation_date                      0            0   
              review_answer_timestamp                   0            0   

                                                      dtype  
orders        order_purchase_timestamp       datetime64[ns]  
              order_approved_at              datetime64[ns]  
              order_delivered_carrier_date   datetime64[ns]  
              order_delivered_customer_date  datetime64[ns]  
              order_estimated_delivery_date  datetime64[ns]  
order_items   shipping_limit_date            datetime64[ns]  
order_reviews review_creation_date           datetime64[ns]  
              review_answer_timestamp        datetime64[ns]